# Zhu2005

https://doi.org/10.1007/s00425-005-0064-4

This notebook demonstrates and validates the Zhu et al. (2005) chlorophyll-fluorescence induction model implemented in `mxlmodels`. The validation compares the model with both panels of manuscript Figure 3: the fluorescence-induction curve and the corresponding proportion of reduced $Q_A$.

In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
from mxlpy import Scipy, Simulator

from mxlmodels import get_zhu_2005

## Load the complete model

The source constructor contains the complete reusable model. Figure-specific reconstruction values are applied only in this validation notebook and are not embedded in `get_zhu2005()`.

In [ ]:
model = get_zhu_2005()
model.parameters

## Figure 3 reconstruction settings

Several quantities required for exact numerical reproduction are not uniquely specified in the manuscript. The settings below were reconstructed from the printed Figure 3 curves while retaining the published reaction network. They are used exclusively for validation and are not presented as the authors' original computational defaults.

In [ ]:
figure3_parameters = {
    "P680Pheo_total": 0.16902,
    "PQ_total": 1.01412,
    "p": 1.0,
    "k1_open": 3.26383e11,
    "k1_closed": 5.22213e10,
    "k01": 133.95,
    "kox": 50.0,
    "kr3": 80.0,
    "kAd": 2.0e8,
    "kUd_closed": 2.0e8,
}

model = model.update_parameters(figure3_parameters)

pool_scale = figure3_parameters["P680Pheo_total"]
initial_conditions = model.get_initial_conditions()

initial_condition_updates = {
    "S0T": 0.2 * pool_scale,
    "S1T": 0.8 * pool_scale,
    "S2T": 0.0,
    "S3T": 0.0,
    "S0Tp": 0.0,
    "S1Tp": 0.0,
    "S2Tp": 0.0,
    "S3Tp": 0.0,
    "QA_QB": pool_scale,
    "QAred_QB": 0.0,
    "QA_QBred": 0.0,
    "QAred_QBred": 0.0,
    "QA_QB2red": 0.0,
    "QAred_QB2red": 0.0,
    "PQH2": 3.0 * pool_scale,
}

for name, value in initial_condition_updates.items():
    initial_conditions[name] = value

## Digitised Figure 3 reference

The reference points below were digitised from the manuscript rendering. Fluorescence is reported in arbitrary units, whereas the reduced-$Q_A$ fraction has an absolute scale from zero to one.

In [ ]:
reference_log_time = np.arange(-8.0, 0.01, 0.5)

reference_fluorescence = np.array(
    [
        0.942,
        0.942,
        0.942,
        0.942,
        0.942,
        0.942,
        0.959,
        1.025,
        1.322,
        4.116,
        5.818,
        6.483,
        7.289,
        7.897,
        7.917,
        7.917,
        7.917,
    ]
)

reference_qa_reduced = np.array(
    [
        0.0000,
        0.0000,
        0.0000,
        0.0010,
        0.0020,
        0.0100,
        0.0560,
        0.1675,
        0.4730,
        0.8961,
        0.9825,
        0.9950,
        1.0000,
        1.0000,
        1.0000,
        1.0000,
        1.0000,
    ]
)

## Simulate Figure 3 conditions

The simulation covers $10^{-8}$ to $10^0$ seconds under an incident photon flux density of 3000 $\mu mol\ photons\ m^{-2}\ s^{-1}$. A stiff BDF integrator is used because excitation transfer and plastoquinone reactions operate on widely separated timescales.

In [ ]:
log_time = np.linspace(-8.0, 0.0, 401)
time = 10.0**log_time

simulation = (
    Simulator(
        model,
        y0=initial_conditions,
        integrator=partial(
            Scipy,
            method="BDF",
            atol=1e-12,
            rtol=1e-8,
        ),
    )
    .simulate_time_course(time)
    .get_result()
    .unwrap_or_err()
    .get_combined()
)

fluorescence = np.interp(
    time,
    simulation.index,
    simulation["F"],
)
qa_reduced = np.interp(
    time,
    simulation.index,
    simulation["QA_reduction_fraction"],
)

## Figure 3 reproduction

Fluorescence is mapped to the arbitrary-unit scale used in the manuscript. The reduced-$Q_A$ curve is plotted directly on its absolute scale and is not rescaled.

In [ ]:
fluorescence_baseline_log_time = -7.0
baseline_index = int(
    np.searchsorted(
        log_time,
        fluorescence_baseline_log_time,
    )
)

fluorescence_for_plot = fluorescence.copy()
fluorescence_for_plot[:baseline_index] = fluorescence[baseline_index]

fluorescence_paper_units = reference_fluorescence[0] + (
    fluorescence_for_plot - fluorescence[baseline_index]
) / (fluorescence[-1] - fluorescence[baseline_index]) * (
    reference_fluorescence[-1] - reference_fluorescence[0]
)

fig, ax_f = plt.subplots(figsize=(6.3, 5.0))
ax_qa = ax_f.twinx()

line_f = ax_f.plot(
    log_time,
    fluorescence_paper_units,
    color="black",
    linewidth=2,
    label="Fluorescence (a)",
)[0]

line_qa = ax_qa.plot(
    log_time,
    qa_reduced,
    color="black",
    linestyle=":",
    marker="v",
    markevery=8,
    markersize=3,
    label="Reduced QA (b)",
)[0]

ax_f.set_xlim(-8, 0)
ax_f.set_ylim(0, 10)
ax_qa.set_ylim(0, 1)
ax_f.set_xlabel("Log Time (s)")
ax_f.set_ylabel("Fluorescence Emission (arbitrary unit)")
ax_qa.set_ylabel("Proportion of reduced QA")
ax_f.grid()
ax_f.legend(
    handles=[line_f, line_qa],
    frameon=False,
    loc="center left",
)
plt.tight_layout()
plt.savefig("zhu2005-fig3.svg")
plt.show()

## Quantitative validation

Fluorescence is O-P normalised before comparison because the manuscript fluorescence axis is arbitrary. The reduced-$Q_A$ fraction is compared directly without rescaling.

In [ ]:
reference_f = np.interp(
    log_time,
    reference_log_time,
    reference_fluorescence,
)
reference_qa = np.interp(
    log_time,
    reference_log_time,
    reference_qa_reduced,
)

comparison = log_time >= fluorescence_baseline_log_time

model_f_normalised = (fluorescence[comparison] - fluorescence[comparison][0]) / (
    fluorescence[comparison][-1] - fluorescence[comparison][0]
)

reference_f_normalised = (reference_f[comparison] - reference_f[comparison][0]) / (
    reference_f[comparison][-1] - reference_f[comparison][0]
)

fluorescence_rmse = np.sqrt(np.mean((model_f_normalised - reference_f_normalised) ** 2))
qa_rmse = np.sqrt(np.mean((qa_reduced - reference_qa) ** 2))

print(f"Normalised fluorescence RMSE: {fluorescence_rmse:.4f}")
print(f"Reduced-QA fraction RMSE: {qa_rmse:.4f}")
print(f"Maximum reduced-QA fraction: {qa_reduced.max():.4f}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

axs[0].plot(
    log_time[comparison],
    reference_f_normalised,
    color="black",
    linewidth=2,
    label="Figure 3a",
)
axs[0].plot(
    log_time[comparison],
    model_f_normalised,
    label="MxlPy",
)
axs[0].set_title("Fluorescence")
axs[0].set_ylabel("Normalised response")

axs[1].plot(
    log_time,
    reference_qa,
    color="black",
    linewidth=2,
    label="Figure 3b",
)
axs[1].plot(
    log_time,
    qa_reduced,
    label="MxlPy",
)
axs[1].set_title("Reduced QA")
axs[1].set_ylim(-0.03, 1.05)

for ax in axs:
    ax.set_xlabel("log10 time [s]")
    ax.legend(frameon=False)

plt.tight_layout()
plt.show()